In [ ]:
# Configuration
TARGET_COLUMN = 'AAPL'  # Stock ticker for Viatris

# Step 1: Import Libraries and Install Dependencies
print("\nStep 1: Installing dependencies...")
!pip install --force-reinstall tensorflow==2.16.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.5.2 optuna==4.0.0 lightgbm==4.5.0 xgboost==2.1.1 pywt==1.5.0 pykalman==0.9.7 transformers==4.44.2 torch==2.4.1 requests==2.32.3 beautifulsoup4==4.12.3
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from google.colab import files
import warnings
warnings.filterwarnings('ignore')
import time
import random
import torch


Step 1: Installing dependencies...
  Using cached tensorflow-2.16.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.3 kB)
  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached scikit_learn-1.5.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached optuna-4.0.0-py3-none-any.whl.metadata (16 kB)
  Using cached lightgbm-4.5.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached xgboost-2.1.1-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the re

In [ ]:
# Optuna
try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
    print("Optuna version:", optuna.__version__)
except Exception:
    print("Optuna installation failed.")
    !pip install optuna==4.0.0
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True

# TensorFlow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, GRU, Dropout, MultiHeadAttention, GlobalAveragePooling1D, Input, Flatten
    from tensorflow.keras.regularizers import l2
    TF_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except Exception:
    print("TensorFlow installation failed.")
    !pip install tensorflow==2.16.1
    import tensorflow as tf
    TF_AVAILABLE = True

# PyWavelets
try:
    import pywt
    PYWT_AVAILABLE = True
except Exception:
    print("PyWavelets installation failed.")
    !pip install pywt==1.5.0
    import pywt
    PYWT_AVAILABLE = True

# PyKalman
try:
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True
except Exception:
    print("PyKalman installation failed.")
    !pip install pykalman==0.9.7
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True

# LightGBM
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

# Transformers and PyTorch
TRANSFORMERS_AVAILABLE = False
try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True
    print("Transformers version:", transformers.__version__)
    print("PyTorch version:", torch.__version__)
except Exception:
    print("Transformers or PyTorch installation failed.")
    !pip install transformers==4.44.2 torch==2.4.1
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True

# Requests and BeautifulSoup
import requests
from bs4 import BeautifulSoup

Optuna version: 4.0.0
TensorFlow version: 2.20.0
Transformers version: 5.13.1
PyTorch version: 2.11.0+cpu


In [ ]:
# Step 2: Load and Clean Data
print("\nStep 2: Loading and cleaning data...")
try:
    uploaded = files.upload()
    stock_data = {}
    news_data = None
    for file_name in uploaded.keys():
        print(f"Processing file: {file_name}")
        if file_name.endswith('.csv'):
            if 'news_data' in file_name.lower():
                try:
                    news_data = pd.read_csv(file_name)
                    print(f"News data loaded: {len(news_data)} rows")
                    print("News data columns:", news_data.columns.tolist())
                    print("News data sample:\n", news_data.head(2))
                except Exception as e:
                    print(f"Error loading news data {file_name}: {e}")
                    news_data = None
            else:
                try:
                    df = pd.read_csv(file_name)
                    print(f"Columns: {df.columns.tolist()}, Rows: {len(df)}")
                    if 'Timestamp' not in df.columns or TARGET_COLUMN not in df.columns:
                        raise ValueError(f"Expected 'Timestamp' and '{TARGET_COLUMN}' columns")
                    stock_data[TARGET_COLUMN] = df
                    stock_data[TARGET_COLUMN][TARGET_COLUMN] = stock_data[TARGET_COLUMN][TARGET_COLUMN].interpolate().ffill().bfill()
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna()
                    stock_data[TARGET_COLUMN]['Timestamp'] = pd.to_datetime(stock_data[TARGET_COLUMN]['Timestamp'], errors='coerce')
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna(subset=['Timestamp'])
                    print(f"Rows after cleaning: {len(stock_data[TARGET_COLUMN])}")
                    print("Price data sample:\n", stock_data[TARGET_COLUMN].head(2))
                except Exception as e:
                    print(f"Error loading price data {file_name}: {e}")
except Exception as e:
    print(f"Error during file upload: {e}")
    exit()


Step 2: Loading and cleaning data...


Saving AAPL.csv to AAPL (4).csv
Processing file: AAPL (4).csv
Columns: ['Timestamp', 'AAPL'], Rows: 8461
Rows after cleaning: 8461
Price data sample:
                   Timestamp        AAPL
0 2026-07-15 11:25:00-04:00  325.320007
1 2026-07-15 11:26:00-04:00  325.214996


In [ ]:
# Step 2.5: Fetch News from FinViz if No News Data Provided
if news_data is None and TRANSFORMERS_AVAILABLE:
    print("\nStep 2.5: No news data uploaded. Fetching news from FinViz...")
    def fetch_finviz_news(ticker):
        url = f'https://finviz.com/quote.ashx?t={ticker}'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            news_table = soup.find('table', {'id': 'news-table'})
            if not news_table:
                print("No news table found on FinViz")
                return None
            news_data = []
            for row in news_table.findAll('tr'):
                cells = row.findAll('td')
                if len(cells) < 2:
                    continue
                headline_link = cells[1].find('a')
                if not headline_link:
                    continue
                headline = headline_link.text.strip()
                link = headline_link['href']
                if link.startswith('/'):
                    link = f'https://finviz.com{link}'
                news_data.append({'Headline': headline, 'Headline_Link': link})
                time.sleep(0.5)
            df = pd.DataFrame(news_data)
            df = df.head(100)  # Limit to last 100 articles
            df['News_Index'] = range(1, len(df) + 1)  # 1 (least recent) to 100 (most recent)
            print(f"Fetched {len(df)} news articles from FinViz")
            return df
        except Exception as e:
            print(f"Error fetching FinViz data for {ticker}: {e}")
            return None

    def fetch_article_content(url, headline):
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36'
        ]
        for attempt in range(3):
            try:
                headers = {'User-Agent': random.choice(user_agents)}
                response = requests.get(url, headers=headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, 'html.parser')
                paragraphs = soup.find_all('p')
                article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
                if len(article_text.strip()) > 50:
                    print(f"Successfully fetched article from {url[:50]}...")
                    return article_text
                else:
                    print(f"Short content from {url[:50]}...")
                    return headline
            except Exception as e:
                print(f"Error fetching article from {url[:50]}...: {e}")
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    return headline
        return headline

    news_data = fetch_finviz_news(TARGET_COLUMN)
    if news_data is not None and not news_data.empty:
        print(f"Fetching article content for {len(news_data)} articles...")
        news_data['News_detail'] = news_data.apply(lambda row: fetch_article_content(row['Headline_Link'], row['Headline']), axis=1)
        news_data['News_detail'] = news_data['News_detail'].apply(lambda x: str(x) if x and len(str(x).strip()) > 0 else None)
        valid_articles = news_data['News_detail'].notna().sum()
        print(f"Fetched {valid_articles} valid articles out of {len(news_data)}")
        if valid_articles == 0:
            print("No valid article content fetched. Disabling sentiment analysis.")
            news_data = None
    else:
        print("Failed to fetch news from FinViz.")


Step 2.5: No news data uploaded. Fetching news from FinViz...
Fetched 100 news articles from FinViz
Fetching article content for 100 articles...
Successfully fetched article from https://finance.yahoo.com/economy/articles/trump-g...
Successfully fetched article from https://finance.yahoo.com/markets/stocks/articles/...
Successfully fetched article from https://finviz.com/news/382248/apple-develops-chin...
Successfully fetched article from https://finance.yahoo.com/economy/articles/taiwan-...
Error fetching article from https://www.barrons.com/articles/berkshire-hathawa...: 401 Client Error: HTTP Forbidden for url: https://www.barrons.com/articles/berkshire-hathaway-alphabet-microsoft-stock-52115df5?mod=bar_FV
Error fetching article from https://www.barrons.com/articles/berkshire-hathawa...: 401 Client Error: HTTP Forbidden for url: https://www.barrons.com/articles/berkshire-hathaway-alphabet-microsoft-stock-52115df5?mod=bar_FV
Error fetching article from https://www.barrons.com/articl

In [ ]:
# Step 2.6: Process News Data with FinBERT
if TRANSFORMERS_AVAILABLE and news_data is not None:
    print("\nStep 2.6: Processing news data with FinBERT...")
    tokenizer = None
    model = None
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
    try:
        torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        model.to(device)
        print(f"FinBERT model loaded on {device}.")
    except Exception as e:
        print(f"Failed to load FinBERT model: {e}. Disabling sentiment analysis.")
        TRANSFORMERS_AVAILABLE = False
        news_data = None

    if news_data is not None:
        print("Summarizing news articles...")
        summarizer = None
        try:
            torch.cuda.empty_cache()
            # Using device=-1 for CPU, which is the default for summarization pipeline
            summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
            print("Summarizer loaded on CPU.")
        except Exception as e:
            print(f"Failed to load summarizer: {e}")
            # Fallback to using headline if summarizer fails
            news_data['Summary'] = news_data['News_detail'].fillna(news_data['Headline'])

        if summarizer:
            def summarize_text(text):
                if not text or pd.isna(text) or len(str(text).strip()) == 0:
                    return ""
                text = str(text)[:1024] # Truncate to avoid exceeding max token limit
                word_count = len(text.split())
                if word_count <= 30: # Summarize only if text is long enough
                    return text
                try:
                    # Adjust max_length and min_length based on input text length
                    max_len = min(150, max(30, int(word_count * 0.3)))
                    min_len = max(10, min(max_len, int(word_count * 0.1)))
                    summary = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False, batch_size=1)[0]['summary_text']
                    return summary
                except Exception as e:
                    print(f"Error summarizing text {text[:50]}...: {e}")
                    return text[:256] # Return truncated text on error

            # Apply summarization only to rows where News_detail is not None and not empty
            news_data['Summary'] = news_data.apply(
                lambda row: summarize_text(row['News_detail']) if pd.notna(row['News_detail']) and len(str(row['News_detail']).strip()) > 0 else row['Headline'],
                axis=1
            )

            unique_summaries = news_data['Summary'].nunique()
            print(f"Generated {unique_summaries} unique summaries out of {len(news_data)}")
            print("News summaries sample:\n", news_data[['Summary']].head(5))

        def get_finbert_sentiment(text):
            if not text or pd.isna(text) or len(str(text).strip()) == 0:
                return np.array([0.33, 0.33, 0.33]) # Return neutral scores for empty or invalid text
            try:
                # Use the text directly for sentiment analysis, truncation handled by tokenizer
                inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
                with torch.no_grad():
                    outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
                return probs.cpu().numpy().flatten()
            except Exception as e:
                print(f"Error analyzing sentiment for text {str(text)[:50]}...: {e}")
                return np.array([0.33, 0.33, 0.33]) # Return neutral scores on error

        # Apply sentiment analysis to the 'Summary' column
        news_data['sentiment_scores'] = news_data['Summary'].apply(get_finbert_sentiment)

        # Expand the sentiment scores into separate columns
        news_data[['positive', 'negative', 'neutral']] = pd.DataFrame(
            news_data['sentiment_scores'].tolist(),
            columns=['positive', 'negative', 'neutral'],
            index=news_data.index
        )
        news_data['polarity'] = news_data['positive'] - news_data['negative']
        news_data['strength'] = news_data[['positive', 'negative', 'neutral']].max(axis=1)
        news_data['weight'] = news_data['News_Index'] / news_data['News_Index'].max() if news_data['News_Index'].max() > 0 else 0 # Normalize weight
        news_data['impact'] = news_data['polarity'] * news_data['strength'] * news_data['weight']
        print("Sentiment scores and impact sample:\n", news_data[['News_Index', 'positive', 'negative', 'neutral', 'polarity', 'strength', 'weight', 'impact']].head(5))

        # Merge news impact with price data
        try:
            if TARGET_COLUMN in stock_data and not stock_data[TARGET_COLUMN].empty:
                print("Aligning news with price data...")
                stock_df = stock_data[TARGET_COLUMN].sort_values('Timestamp').reset_index(drop=True)
                n_news = len(news_data)
                n_prices = len(stock_df)

                # Ensure news_data has the same number of rows as stock_df for direct merging
                if n_news != n_prices:
                    print(f"Mismatch between news ({n_news}) and price ({n_prices}) data rows. Resampling news data...")
                    # Create a dummy DataFrame for news with the same index as stock_df
                    aligned_news_data = pd.DataFrame(index=stock_df.index, columns=news_data.columns)

                    # Assign news data to the aligned DataFrame, repeating news if necessary
                    news_indices = np.arange(n_news)
                    aligned_news_data.iloc[:min(n_news, n_prices)] = news_data.iloc[:min(n_news, n_prices)].values

                    # If there are more price data points than news, forward fill with the last news sentiment
                    if n_prices > n_news:
                         aligned_news_data.iloc[n_news:] = news_data.iloc[-1].values

                    # If there are more news data points than price data points, truncate news
                    if n_news > n_prices:
                         aligned_news_data = aligned_news_data.iloc[:n_prices]

                    news_data = aligned_news_data

                # Ensure columns match before assignment
                news_sentiment_cols = ['positive', 'negative', 'neutral', 'polarity', 'strength', 'impact']
                if all(col in news_data.columns for col in news_sentiment_cols):
                    merged_data = stock_df.copy()
                    merged_data[news_sentiment_cols] = news_data[news_sentiment_cols].values
                    # Fill any remaining NaNs after alignment with neutral sentiment
                    merged_data[news_sentiment_cols] = merged_data[news_sentiment_cols].fillna(0.33)
                    stock_data[TARGET_COLUMN] = merged_data
                    print(f"Merged data shape: {stock_data[TARGET_COLUMN].shape}")
                    print("Merged data sample:\n", stock_data[TARGET_COLUMN].head(5))
                else:
                     print("Sentiment columns not found in news data after alignment. Proceeding without sentiment features.")


        except Exception as e:
            print(f"Error merging news and price data: {e}. Proceeding without sentiment features.")
            news_data = None
else:
    print("No news data or Transformers unavailable.")


Step 2.6: Processing news data with FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (2 outputs left out)

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT model loaded on cuda.
Summarizing news articles...


config.json: 0.00B [00:00, ?B/s]

Failed to load summarizer: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"
Sentiment scores and impact sample:
    News_Index  positive  negative   neutral  polarity  strength  weight  \
0           1  0.769126  0.008499  0.222375  0.760628  0.769126    0.01   
1           2  

In [ ]:
# Step 3: Denoise Data for Highest SNR
print("\nStep 3: Denoising data...")
price_scaler = MinMaxScaler()
feature_columns = [TARGET_COLUMN]
if TRANSFORMERS_AVAILABLE and news_data is not None:
    feature_columns.extend(['positive', 'negative', 'neutral', 'polarity', 'strength', 'impact'])
if TARGET_COLUMN in stock_data and not stock_data[TARGET_COLUMN].empty:
    scaled_data = price_scaler.fit_transform(stock_data[TARGET_COLUMN][[TARGET_COLUMN]])
    print(f"Scaled data shape: {scaled_data.shape}")

    def calculate_snr(original, denoised):
        noise = original - denoised
        signal_power = np.mean(original ** 2)
        noise_power = np.var(noise)
        return 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')

    def optimize_denoising(trial, data):
        methods = []
        if PYWT_AVAILABLE:
            wavelet = trial.suggest_categorical('wavelet', ['db1', 'db2', 'sym2'])
            level = trial.suggest_int('wavelet_level', 1, 3)
            wavelet_denoised = pywt.wavedec(data, wavelet, level=level)
            threshold = np.std(wavelet_denoised[-1]) * np.sqrt(2 * np.log(len(data)))
            wavelet_denoised = pywt.waverec([pywt.threshold(c, threshold, mode='soft') for c in wavelet_denoised], wavelet)[:len(data)]
            snr_wavelet = calculate_snr(data, wavelet_denoised)
            methods.append(('wavelet', snr_wavelet, wavelet_denoised))
        ma_window = trial.suggest_int('ma_window', 3, 10)
        ma_denoised = pd.Series(data).rolling(window=ma_window, min_periods=1).mean().values
        snr_ma = calculate_snr(data, ma_denoised)
        methods.append(('ma', snr_ma, ma_denoised))
        if TF_AVAILABLE:
            encoding_dim = trial.suggest_int('encoding_dim', 16, 32)
            model = Sequential([Dense(encoding_dim, activation='relu', input_shape=(1,)),
                                Dense(1, activation='linear')])
            model.compile(optimizer='adam', loss='mse')
            autoencoder_denoised = model.predict(data.reshape(-1, 1), verbose=0).flatten()
            snr_autoencoder = calculate_snr(data, autoencoder_denoised)
            methods.append(('autoencoder', snr_autoencoder, autoencoder_denoised))
        if PYKALMAN_AVAILABLE:
            kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], initial_state_mean=data[0])
            kalman_denoised, _ = kf.filter(data)
            snr_kalman = calculate_snr(data, kalman_denoised.flatten())
            methods.append(('kalman', snr_kalman, kalman_denoised))
        best_method = max(methods, key=lambda x: x[1] if not np.isinf(x[1]) else -float('inf'))
        return best_method[1], best_method[2]

    if OPTUNA_AVAILABLE:
        print("Optimizing denoising with Optuna...")
        study_denoise = optuna.create_study(direction='maximize', sampler=TPESampler())
        study_denoise.optimize(lambda trial: optimize_denoising(trial, scaled_data[:, 0])[0], n_trials=10)
        _, denoised_data = optimize_denoising(optuna.trial.FixedTrial(study_denoise.best_params), scaled_data[:, 0])
        print(f"Best SNR after Optuna: {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")
    else:
        print("Optuna unavailable. Using default moving average denoising...")
        denoised_data = pd.Series(scaled_data[:, 0]).rolling(window=5, min_periods=1).mean().values
        print(f"Best SNR (default): {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")

    if TRANSFORMERS_AVAILABLE and news_data is not None:
        stock_denoised_data = {TARGET_COLUMN: stock_data[TARGET_COLUMN][feature_columns].copy()}
        stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN] = denoised_data
    else:
        stock_denoised_data = {TARGET_COLUMN: pd.DataFrame({TARGET_COLUMN: denoised_data})}
else:
    print("No stock data available. Exiting.")
    exit()

[I 2026-05-01 08:36:37,027] A new study created in memory with name: no-name-b00938dc-fa70-41c6-be30-096f9c32c539



Step 3: Denoising data...
Scaled data shape: (8190, 1)
Optimizing denoising with Optuna...


[I 2026-05-01 08:36:41,882] Trial 0 finished with value: 50.53104354427382 and parameters: {'wavelet': 'db1', 'wavelet_level': 1, 'ma_window': 6, 'encoding_dim': 23}. Best is trial 0 with value: 50.53104354427382.
[I 2026-05-01 08:36:44,209] Trial 1 finished with value: 49.620800048453745 and parameters: {'wavelet': 'sym2', 'wavelet_level': 3, 'ma_window': 10, 'encoding_dim': 17}. Best is trial 0 with value: 50.53104354427382.
[I 2026-05-01 08:36:46,589] Trial 2 finished with value: 49.620800048453745 and parameters: {'wavelet': 'db1', 'wavelet_level': 2, 'ma_window': 7, 'encoding_dim': 16}. Best is trial 0 with value: 50.53104354427382.
[I 2026-05-01 08:36:48,891] Trial 3 finished with value: 49.620800048453745 and parameters: {'wavelet': 'db1', 'wavelet_level': 3, 'ma_window': 8, 'encoding_dim': 29}. Best is trial 0 with value: 50.53104354427382.
[I 2026-05-01 08:36:51,156] Trial 4 finished with value: 49.620800048453745 and parameters: {'wavelet': 'sym2', 'wavelet_level': 3, 'ma_win

Best SNR after Optuna: 50.53 dB


In [ ]:
# Step 4: Prepare Sequences
print("\nStep 4: Preparing sequences...")
SEQ_LENGTH = 30
WINDOW_SIZE = 300
stock_X_deep = {}
stock_X_ml = {}
stock_y = {}
full_data = stock_denoised_data[TARGET_COLUMN][feature_columns].values
X_for_prediction = np.array([full_data[i:i + SEQ_LENGTH] for i in range(len(full_data) - SEQ_LENGTH)])
stock_y[TARGET_COLUMN] = stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN].values[SEQ_LENGTH:]
stock_X_deep[TARGET_COLUMN] = X_for_prediction
stock_X_ml[TARGET_COLUMN] = X_for_prediction
print(f"X_deep shape: {stock_X_deep[TARGET_COLUMN].shape}, y shape: {stock_y[TARGET_COLUMN].shape}")
last_300_scaled = stock_denoised_data[TARGET_COLUMN][feature_columns].values[-WINDOW_SIZE:]


Step 4: Preparing sequences...
X_deep shape: (8160, 30, 7), y shape: (8160,)


In [ ]:
# Step 5: Define and Optimize Models
print("\nStep 5: Defining and optimizing models...")
feature_dim = len(feature_columns)
def optimize_cnn_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_conv1d_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_bilstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Bidirectional(LSTM(lstm_units, kernel_regularizer=l2(0.02))),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_stacked_lstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_gru(trial):
    gru_units = trial.suggest_int('gru_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        GRU(gru_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_lightgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 1.0)
    }
    return lgb.LGBMRegressor(**params, force_col_wise=True)

def optimize_xgboost(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0)
    }
    return xgb.XGBRegressor(**params)

def optimize_transformer(trial):
    num_heads = trial.suggest_int('num_heads', 4, 8)
    ff_dim = trial.suggest_int('ff_dim', 64, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(inputs, inputs)
    x = Dropout(dropout_rate)(x)
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(x, x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(ff_dim, activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_tcn(trial):
    filters = trial.suggest_int('filters', 32, 64)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = inputs
    shortcut = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    shortcut = Conv1D(filters=filters, kernel_size=1, padding='same')(shortcut)
    x = tf.keras.layers.Add()([shortcut, x])
    x = tf.keras.layers.Activation('relu')(x)
    x = Flatten()(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(0.02))(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model


Step 5: Defining and optimizing models...


In [ ]:
models = [
    ('cnn_lstm', optimize_cnn_lstm, 'deep'),
    ('conv1d_lstm', optimize_conv1d_lstm, 'deep'),
    ('bilstm', optimize_bilstm, 'deep'),
    ('stacked_lstm', optimize_stacked_lstm, 'deep'),
    ('gru', optimize_gru, 'deep'),
    ('lightgbm', optimize_lightgbm, 'ml'),
    ('xgboost', optimize_xgboost, 'ml'),
    ('transformer', optimize_transformer, 'deep'),
    ('tcn', optimize_tcn, 'deep')
]

# Step 6: Train and Evaluate Models with Walk-Forward Validation
print("\nStep 6: Training and evaluating models...")
def walk_forward_validation(model, model_name, X, y, price_scaler, step_size=300, max_steps=20):
    best_adj_r2 = -float('inf')
    best_model = None
    adj_r2_history = []
    n_steps = min(max_steps, max(1, (len(X) - SEQ_LENGTH) // step_size))
    if n_steps < 1:
        print(f"Error: Insufficient data for walk-forward validation for {model_name}.")
        return {'adj_r2': 0, 'best_model': model}, adj_r2_history
    n_predictors = feature_dim
    for i in range(1, n_steps + 1):
        train_end = i * step_size
        test_size = min(step_size, len(X) - train_end)
        train_X = X[:train_end]
        train_y = y[:train_end]
        test_X = X[train_end:train_end + test_size]
        test_y = y[train_end:train_end + test_size]
        if len(test_X) == 0 or len(test_y) == 0 or len(test_X) != len(test_y):
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                model.fit(train_X[:, :, 0].reshape(-1, SEQ_LENGTH), train_y)
                preds = model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
            else:
                model.fit(train_X, train_y, epochs=10, batch_size=64, validation_split=0.2, verbose=0,
                          callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
                preds = model.predict(test_X, verbose=0).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
            ss_res = np.sum((test_y_inv - preds_inv) ** 2)
            ss_tot = np.sum((test_y_inv - np.mean(test_y_inv)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(test_y_inv)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_predictors - 1) if n > n_predictors + 1 else 0
            adj_r2_history.append(adj_r2)
            if adj_r2 > best_adj_r2:
                best_adj_r2 = adj_r2
                best_model = tf.keras.models.clone_model(model) if model_name not in ['xgboost', 'lightgbm'] else model
                if model_name not in ['xgboost', 'lightgbm']:
                    best_model.set_weights(model.get_weights())
        except Exception as e:
            print(f"Error in walk-forward validation for {model_name} at step {i}: {e}")
            continue
    return {'adj_r2': best_adj_r2, 'best_model': best_model}, adj_r2_history


Step 6: Training and evaluating models...


In [ ]:
stock_results = {}
for stock_name in stock_data.keys():
    results = {}
    for model_name, model_fn, data_type in models:
        if not (TF_AVAILABLE and data_type == 'deep') and not (LGB_AVAILABLE and model_name == 'lightgbm') and not (XGB_AVAILABLE and model_name == 'xgboost'):
            continue
        print(f"\nTraining {model_name} for {stock_name}...")
        X_data = stock_X_deep[stock_name] if data_type == 'deep' else stock_X_ml[stock_name]
        try:
            if OPTUNA_AVAILABLE:
                study = optuna.create_study(direction='minimize', sampler=TPESampler())
                def objective(trial):
                    model = model_fn(trial)
                    n_samples = len(X_data)
                    train_size = int(0.8 * n_samples)
                    train_X, train_y = X_data[:train_size], stock_y[stock_name][:train_size]
                    val_X, val_y = X_data[train_size:], stock_y[stock_name][train_size:]
                    if model_name in ['xgboost', 'lightgbm']:
                        model.fit(train_X[:, :, 0].reshape(-1, SEQ_LENGTH), train_y)
                        preds = model.predict(val_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                    else:
                        model.fit(train_X, train_y, epochs=5, batch_size=64, validation_split=0.2, verbose=0,
                                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                        preds = model.predict(val_X, verbose=0).flatten()
                    val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
                    preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
                    ss_res = np.sum((val_y_inv - preds_inv) ** 2)
                    ss_tot = np.sum((val_y_inv - np.mean(val_y_inv)) ** 2)
                    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                    n = len(val_y_inv)
                    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                    return -adj_r2
                study.optimize(objective, n_trials=3)
                model = model_fn(optuna.trial.FixedTrial(study.best_params))
            else:
                model = model_fn({})
            result, adj_r2_history = walk_forward_validation(model, model_name, X_data, stock_y[stock_name], price_scaler)
            results[model_name] = result
            print(f"{model_name} - {stock_name} - Best Adj R²: {result['adj_r2']:.6f}")
        except Exception as e:
            print(f"Error training {model_name} for {stock_name}: {e}")
            results[model_name] = {'adj_r2': 0, 'best_model': None}
    stock_results[stock_name] = results

[I 2026-05-01 08:37:23,812] A new study created in memory with name: no-name-be9c8243-48bc-4fa5-9941-5f161434ca11



Training cnn_lstm for NVDA...


[I 2026-05-01 08:37:57,008] Trial 0 finished with value: 0.7314892011495204 and parameters: {'filters': 32, 'lstm_units': 98, 'dropout_rate': 0.4503056415106724, 'learning_rate': 0.0027829135955543083}. Best is trial 0 with value: 0.7314892011495204.
[I 2026-05-01 08:38:08,892] Trial 1 finished with value: 1.0489923764215643 and parameters: {'filters': 48, 'lstm_units': 63, 'dropout_rate': 0.5938508346162676, 'learning_rate': 0.002768334480398244}. Best is trial 0 with value: 0.7314892011495204.
[I 2026-05-01 08:38:21,494] Trial 2 finished with value: 0.2995037223757162 and parameters: {'filters': 50, 'lstm_units': 121, 'dropout_rate': 0.5638348319907042, 'learning_rate': 0.0012352538537247313}. Best is trial 2 with value: 0.2995037223757162.
[I 2026-05-01 08:41:07,955] A new study created in memory with name: no-name-e4ff6c55-3d44-487e-a7f6-f5967329bc28


cnn_lstm - NVDA - Best Adj R²: -0.197707

Training conv1d_lstm for NVDA...


[I 2026-05-01 08:41:16,279] Trial 0 finished with value: -0.597535356791776 and parameters: {'filters': 35, 'lstm_units': 56, 'dropout_rate': 0.35023651247500204, 'learning_rate': 0.003087990404959154}. Best is trial 0 with value: -0.597535356791776.
[I 2026-05-01 08:41:24,711] Trial 1 finished with value: -0.6835541670282317 and parameters: {'filters': 64, 'lstm_units': 126, 'dropout_rate': 0.48766333053206623, 'learning_rate': 0.00040206270024700444}. Best is trial 1 with value: -0.6835541670282317.
[I 2026-05-01 08:41:32,607] Trial 2 finished with value: -0.791738696826553 and parameters: {'filters': 42, 'lstm_units': 81, 'dropout_rate': 0.35746250759858456, 'learning_rate': 0.0013982670275407467}. Best is trial 2 with value: -0.791738696826553.
[I 2026-05-01 08:43:29,979] A new study created in memory with name: no-name-cf29939d-8339-411e-a98a-df03cbd0e26c


conv1d_lstm - NVDA - Best Adj R²: 0.671573

Training bilstm for NVDA...


... (16 outputs left out)

[I 2026-05-01 08:53:52,139] Trial 0 finished with value: 2.09685929548522 and parameters: {'num_heads': 6, 'ff_dim': 95, 'dropout_rate': 0.5675190022945442, 'learning_rate': 0.00024980002856682236}. Best is trial 0 with value: 2.09685929548522.
[I 2026-05-01 08:54:07,451] Trial 1 finished with value: -0.8643589150213128 and parameters: {'num_heads': 7, 'ff_dim': 100, 'dropout_rate': 0.5994200405295661, 'learning_rate': 0.0041725805049063844}. Best is trial 1 with value: -0.8643589150213128.
[I 2026-05-01 08:54:22,742] Trial 2 finished with value: -0.7385312582581691 and parameters: {'num_heads': 5, 'ff_dim': 122, 'dropout_rate': 0.3318335602941243, 'learning_rate': 0.003461299067226477}. Best is trial 1 with value: -0.8643589150213128.
[I 2026-05-01 08:55:44,115] A new study created in memory with name: no-name-8c833aa5-f8e2-4784-a708-388a0bb7d82a


transformer - NVDA - Best Adj R²: -0.366658

Training tcn for NVDA...


[I 2026-05-01 08:55:54,464] Trial 0 finished with value: -0.16665122694242518 and parameters: {'filters': 49, 'kernel_size': 2, 'dropout_rate': 0.5906854009287666, 'learning_rate': 0.0030052276978916295}. Best is trial 0 with value: -0.16665122694242518.
[I 2026-05-01 08:56:04,321] Trial 1 finished with value: 1.9719076916761158 and parameters: {'filters': 57, 'kernel_size': 5, 'dropout_rate': 0.3595319381603518, 'learning_rate': 0.004954479026388787}. Best is trial 0 with value: -0.16665122694242518.
[I 2026-05-01 08:56:14,652] Trial 2 finished with value: 0.3045031948104515 and parameters: {'filters': 38, 'kernel_size': 4, 'dropout_rate': 0.31533699826285383, 'learning_rate': 0.00110102991541724}. Best is trial 0 with value: -0.16665122694242518.


tcn - NVDA - Best Adj R²: 0.830896


In [ ]:
# Step 6.5: Analyze Overfitting
print("\nStep 6.5: Analyzing overfitting...")
for stock_name in stock_results.keys():
    results = stock_results[stock_name]
    X_data = stock_X_deep[stock_name] if 'cnn_lstm' in results else stock_X_ml[stock_name]
    y_data = stock_y[stock_name]
    n_samples = len(X_data)
    train_size = int(0.8 * n_samples)
    train_X, val_X = X_data[:train_size], X_data[train_size:]
    train_y, val_y = y_data[:train_size], y_data[train_size:]

    for model_name, result in results.items():
        best_model = result['best_model']
        if best_model is None:
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                train_pred = best_model.predict(train_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                val_pred = best_model.predict(val_X[:, :, 0].reshape(-1, SEQ_LENGTH))
            else:
                best_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
                best_model.fit(train_X, train_y, epochs=5, batch_size=64, verbose=0,
                              callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                train_pred = best_model.predict(train_X, verbose=0).flatten()
                val_pred = best_model.predict(val_X, verbose=0).flatten()

            train_y_inv = price_scaler.inverse_transform(train_y.reshape(-1, 1)).flatten()
            val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
            train_pred_inv = price_scaler.inverse_transform(train_pred.reshape(-1, 1)).flatten()
            val_pred_inv = price_scaler.inverse_transform(val_pred.reshape(-1, 1)).flatten()

            def calculate_adj_r2(y_true, y_pred):
                ss_res = np.sum((y_true - y_pred) ** 2)
                ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
                r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                n = len(y_true)
                adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                return adj_r2

            train_adj_r2 = calculate_adj_r2(train_y_inv, train_pred_inv)
            val_adj_r2 = calculate_adj_r2(val_y_inv, val_pred_inv)
            overfitting_gap = train_adj_r2 - val_adj_r2

            stock_results[stock_name][model_name]['val_adj_r2'] = val_adj_r2
            stock_results[stock_name][model_name]['overfitting_gap'] = overfitting_gap
            print(f"Model: {model_name} for {stock_name} - Val Adj R²: {val_adj_r2:.6f}, Overfitting Gap: {overfitting_gap:.6f}")
            print(f"  Training Adj R²: {train_adj_r2:.6f}")
            if overfitting_gap > 0.2:
                print("  Warning: Potential overfitting detected")
            elif overfitting_gap > 0.1:
                print("  Caution: Moderate overfitting risk")
            else:
                print("  Status: Low overfitting risk")
        except Exception as e:
            print(f"Error analyzing overfitting for {model_name}: {e}")


Step 6.5: Analyzing overfitting...
Model: cnn_lstm for NVDA - Val Adj R²: 0.791713, Overfitting Gap: 0.199952
  Training Adj R²: 0.991665
  Caution: Moderate overfitting risk
Model: conv1d_lstm for NVDA - Val Adj R²: 0.956742, Overfitting Gap: 0.038707
  Training Adj R²: 0.995449
  Status: Low overfitting risk
Model: bilstm for NVDA - Val Adj R²: 0.929104, Overfitting Gap: 0.066322
  Training Adj R²: 0.995426
  Status: Low overfitting risk
Model: stacked_lstm for NVDA - Val Adj R²: 0.794056, Overfitting Gap: 0.196555
  Training Adj R²: 0.990611
  Caution: Moderate overfitting risk
Model: gru for NVDA - Val Adj R²: 0.980481, Overfitting Gap: 0.015721
  Training Adj R²: 0.996202
  Status: Low overfitting risk
Model: lightgbm for NVDA - Val Adj R²: -1.472480, Overfitting Gap: 2.459474
  Training Adj R²: 0.986994
Model: xgboost for NVDA - Val Adj R²: -1.710583, Overfitting Gap: 2.695887
  Training Adj R²: 0.985304
Model: transformer for NVDA - Val Adj R²: 0.763414, Overfitting Gap: 0.2093

In [ ]:
# Step 7: Generate Predictions with Best Non-Overfit Model
print("\nStep 7: Generating predictions...")
best_val_adj_r2 = -float('inf')
best_model_name = None
best_model = None

for stock_name in stock_results.keys():
    results = stock_results[stock_name]
    for model_name, result in results.items():
        if result.get('val_adj_r2', -float('inf')) > best_val_adj_r2 and result.get('overfitting_gap', float('inf')) <= 0.2:
            best_val_adj_r2 = result['val_adj_r2']
            best_model_name = model_name
            best_model = result['best_model']

if best_model_name is None:
    print("No model with acceptable overfitting gap found. Using best overall Adj R².")
    best_model_name = max(results.items(), key=lambda x: x[1]['adj_r2'])[0]
    best_model = results[best_model_name]['best_model']

overfitting_gap = results[best_model_name].get('overfitting_gap', 'N/A')
print(f"Best model: {best_model_name} with Val Adj R²: {best_val_adj_r2:.6f}, Overfitting Gap: {overfitting_gap}")

def generate_predictions(model, model_name, scaled_data, price_scaler, stock_name, timestamps, steps=150):
    preds_scaled = []
    current_sequence = scaled_data[:SEQ_LENGTH].reshape(1, SEQ_LENGTH, feature_dim)
    for _ in range(steps):
        try:
            if model_name in ['xgboost', 'lightgbm']:
                pred_scaled = model.predict(current_sequence[:, :, 0].reshape(1, -1))[0]
            else:
                pred_scaled = model.predict(current_sequence, verbose=0)[0, 0]
            preds_scaled.append(float(pred_scaled))
            current_sequence = np.roll(current_sequence, -1, axis=1)
            current_sequence[0, -1, 0] = pred_scaled
            if feature_dim > 1:
                current_sequence[0, -1, 1:] = current_sequence[0, -2, 1:]
        except Exception as e:
            print(f"Prediction failed for {model_name}: {e}")
            break
    preds_scaled = np.array(preds_scaled).reshape(-1, 1)
    predictions = price_scaler.inverse_transform(preds_scaled).flatten()
    last_timestamp = pd.to_datetime(timestamps.iloc[-1])
    future_timestamps = pd.date_range(start=last_timestamp + pd.Timedelta(minutes=1), periods=steps, freq='T')
    return predictions, future_timestamps

predictions, future_timestamps = generate_predictions(best_model, best_model_name, last_300_scaled, price_scaler, stock_name, stock_data[TARGET_COLUMN]['Timestamp'])
print(f"Predictions (first 5): {predictions[:5]}")
print(f"Future timestamps (first 5): {future_timestamps[:5]}")


Step 7: Generating predictions...
Best model: gru with Val Adj R²: 0.980481, Overfitting Gap: 0.01572060960035182
Predictions (first 5): [201.24022182 201.18426742 201.1815978  201.22604484 201.3007135 ]
Future timestamps (first 5): DatetimeIndex(['2026-04-30 20:00:00+00:00', '2026-04-30 20:01:00+00:00',
               '2026-04-30 20:02:00+00:00', '2026-04-30 20:03:00+00:00',
               '2026-04-30 20:04:00+00:00'],
              dtype='datetime64[ns, UTC]', freq='min')


In [ ]:
# Step 8: Generate Predictions for All Valid Models
print("\nStep 8: Generating predictions for all models with Adj R² > 0.8...")
prediction_results = {}
for stock_name in stock_results.keys():
    valid_models = {}
    for model_name, result in stock_results[stock_name].items():
        if result.get('val_adj_r2', 0) > 0.8 and result.get('overfitting_gap', float('inf')) <= 0.2:
            valid_models[model_name] = result['best_model']
            print(f"Model '{model_name}' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)")

    if not valid_models:
        print(f"No valid models found for stock {stock_name}")
        continue

    predictions_df = pd.DataFrame()
    for model_name, model in valid_models.items():
        print(f"\nGenerating predictions for {model_name}...")
        predictions, future_timestamps = generate_predictions(model, model_name, last_300_scaled, price_scaler, stock_name, stock_data[stock_name]['Timestamp'])
        predictions_df[model_name] = predictions
    predictions_df['Timestamp'] = future_timestamps
    prediction_results[stock_name] = predictions_df

for stock_name, predictions_df in prediction_results.items():
    print(f"\nPredictions for {stock_name}:")
    print(predictions_df.head())
    filename = f'{stock_name}_predictions.csv'
    predictions_df.to_csv(filename, index=False)
    print(f"\nDownloading {filename}...")
    files.download(filename)


Step 8: Generating predictions for all models with Adj R² > 0.8...
Model 'conv1d_lstm' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'bilstm' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'gru' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'tcn' meets criteria (Val Adj R² > 0.8, Overfitting Gap <= 0.2)

Generating predictions for conv1d_lstm...

Generating predictions for bilstm...

Generating predictions for gru...

Generating predictions for tcn...

Predictions for NVDA:
   conv1d_lstm      bilstm         gru         tcn                 Timestamp
0   201.652651  200.499235  201.240222  200.948014 2026-04-30 20:00:00+00:00
1   201.621566  200.454140  201.184267  200.920301 2026-04-30 20:01:00+00:00
2   201.597843  200.402881  201.181598  200.892017 2026-04-30 20:02:00+00:00
3   201.586686  200.334179  201.226045  200.871136 2026-04-30 20:03:00+00:00
4   201.577321  200.260603  201.300713  200.849919 2026-04-30 20:04:00+00:00



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>